# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Can a page's staleness, visibility, and search position be combined into a priority score that tells a content editor which pages to review first for a refresh?

**Decision this supports:** Which page an editor should review and refresh first, given limited time. A wrong pick can wasted the editor time (on a page that didn't need attention) or a genuinely declining page going unreviewed.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** The small anonymized starter dataset (data/raw/content_refresh_anonymized.csv, 30,000 rows) for baseline, modeling, and validation. A March 2026 slice of the FlyRank warehouse (fact_content_daily_performance, ~9.8M rows via Hugging Face + DuckDB) was used separately to verify the data contract.

**Date windows:** Starter dataset covers a trailing-90-day window. Warehouse slice: March 1–31, 2026.

**Excluded, and why:** Any FlyRank product score (health_score, priority_score) using these as features would mean copying an existing decision instead of learning from raw signals. trend_direction/trend_pct were used only to build the label, never as features (leakage risk).

**Public-safe:** No client names, domains, or URLs appear anywhere all IDs are pseudonymous hash codes.

In [ ]:
import os
print(os.getcwd())
os.chdir("..")   

In [5]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** Staleness and visibility are more useful when looked at together than when looked at separately. Visibility has clearly been shown to be useful, but it is still unclear whether staleness alone is a reliable signal.

**Features (5):** days_since_last_update, impressions_90d, avg_position, word_count, search_volume, all decision-time-available, none derived from the label.

**Label/proxy:** is_declining_label = (trend_direction == "down") a same-window proxy, not a validated future outcome.

**Baseline:** stale × visible × impressions_90d, reason code stale_visible_page.

**Validation design:** Client-grouped holdout (GroupShuffleSplit, 25 train / 7 test clients) chosen so the model is tested on clients it has never seen, avoiding client-specific memorization.

**Leakage checks:** Confirmed no label-derived or future-window fields were used as features; a deliberate leakage demo (adding a label-derived column) inflated a correlation from -0.011 to 0.644, showing why this matters.

In [6]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np

features = ["days_since_last_update", "impressions_90d", "avg_position", "word_count", "search_volume"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

X_train, y_train = train_df[features].fillna(0), train_df["is_declining_label"]
X_test, y_test = test_df[features].fillna(0), test_df["is_declining_label"]

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)
test_df["model_prob"] = model.predict_proba(X_test)[:, 1]

test_stale = (test_df["days_since_last_update"] >= 180).astype(int)
test_visible = (test_df["impressions_90d"] >= 500).astype(int)
test_df["baseline_score"] = test_stale * test_visible * test_df["impressions_90d"]

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [7]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = []
for k in (20, 50):
    results.append({
        "k": k,
        "baseline_precision": round(precision_at_k(test_df["baseline_score"], y_test, k), 3),
        "model_precision": round(precision_at_k(test_df["model_prob"], y_test, k), 3),
    })
pd.DataFrame(results)

,k,baseline_precision,model_precision
0,20,0.8,0.65
1,50,0.7,0.62


## 5. Limitations

*What this work cannot claim.*

This work cannot claim to predict Google's ranking algorithm, and cannot prove that refreshing a page causes recovery that would require a controlled experiment. The label is a same-window proxy, not a validated future outcome. Results are measured on a 30,000-row anonymized sample from a limited client set, under one train/test split ,directional, not guaranteed to reproduce exactly on new data. Model precision is least reliable on low-visibility pages specifically (error analysis, Week 5).

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:
df["final_score"] = 0.70 * df["baseline_score"].rank(pct=True) + 0.30 * model.predict_proba(df[features].fillna(0))[:, 1]

def reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page", "review_for_refresh"
    return "low_priority", "no_action"

df[["reason_code", "action"]] = df.apply(lambda r: pd.Series(reason(r)), axis=1)
queue = df.sort_values("final_score", ascending=False)
queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,is_declining_label,final_score,reason_code,action
7452,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.0,moderate,page_1,down,-22.4,821,1,0.937726,stale_visible_page,review_for_refresh
26840,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.0,moderate,page_1,down,-44.5,954,1,0.934155,stale_visible_page,review_for_refresh
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3118.0,20396.0,...,0.0,moderate,striking,down,-45.7,1697,1,0.914126,stale_visible_page,review_for_refresh
22872,content_e3ff1b093148,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,33575.0,...,0.0,moderate,page_1,down,-68.5,1408,1,0.913265,stale_visible_page,review_for_refresh
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4020.0,26513.0,...,0.0,moderate,striking,down,-55.0,828,1,0.912915,stale_visible_page,review_for_refresh
11630,content_6226ee6adc91,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3950.0,27607.0,...,0.0,moderate,striking,down,-28.5,545,1,0.912869,stale_visible_page,review_for_refresh
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3388.0,21742.0,...,0.0,good,striking,down,-52.2,4556,1,0.912251,stale_visible_page,review_for_refresh
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3696.0,24643.0,...,0.0,moderate,page_3_5,stable,-11.7,1316,0,0.912183,stale_visible_page,review_for_refresh
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,7812,1,0.911250,stale_visible_page,review_for_refresh
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4486.0,29333.0,...,0.0,good,page_3_5,down,-74.4,4429,1,0.911122,stale_visible_page,review_for_refresh


Pages are scored with (0.70 × baseline_rank + 0.30 × model_probability), weighted toward the baseline since it validated better. Editors shouldreview review_for_refresh pages first these carry the strongest, validated signal. No page should be auto-refreshed or auto-published based on this score alone.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [9]:
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

import json
metrics = {"baseline_p20": 0.80, "model_p20": 0.65, "baseline_p50": 0.70, "model_p50": 0.62}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved queue and metrics for the paper.")

Saved queue and metrics for the paper.


**ML-12 done in this notebook's closing cells**

**5-minute demo outline:**
1. Problem: editors can't review every page manually (30s)
2. Show baseline rule + its top-20 (1 min)
3. Show model, honest client-grouped result, baseline wins (1.5 min)
4. Show before/after: naive vs honest split inflating results (1 min)
5. Final blended action queue + reason codes (1 min)

**Social-post cut (1-2 sentences):**
> I built a content-refresh priority system for FlyRank's SEO data and discovered my "smarter" 
> ML model actually lost to a simple hand-written rule once I validated it honestly. Sometimes the baseline wins. 

**Employer-facing summary (3 sentences):**
> Built an end-to-end content prioritization pipeline on real search data: a transparent 
> baseline rule, a Random Forest model, and honest client-grouped validation. Discovered and documented that the simple rule outperformed the ML model a finding that shaped the final 
> blended recommendation system. Delivered a public research paper with full reproducibility, 
> leakage audits, and decision-support (not automated) recommendations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
